# Hough Transform in Directional Parameter Space (a-b Domain)

Most standard Hough implementations (e.g., OpenCV or scikit-image) operate in polar parameter space $(\rho, \theta)$. This notebook presents an alternative low-level approach in which the accumulator space is based on the classical line equation:

$$y = ax + b$$

Topics covered:
- Building and indexing the accumulator matrix $H$.
- Mapping pixels from the image domain into parameterized $(a, b)$ space.
- Analyzing mathematical limitations and numerical instability of the algebraic model.

In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np
from skimage.transform import hough_line, hough_line_peaks

def hab(img, amin, amax, ask, bmin, bmax, bsk):
    """
    Low-level construction of the accumulator matrix in directional form.
    """
    A = np.arange(amin, amax, ask)
    B = np.arange(bmin, bmax, bsk)
    H = np.zeros((len(A), len(B)), dtype=np.uint8)
    
    ys, xs = np.nonzero(img)
    
    # Map each active pixel into (a, b) space
    for x, y in zip(xs, ys):
        for i, a in enumerate(A):
            b = y - a * x
            # Quantize and find the nearest grid index
            j = np.argmin(np.abs(B - b))
            H[i, j] += 1
            
    return H

# Generate synthetic test data (2 Dirac impulses)
im = np.zeros((64, 64), dtype=np.uint8)
im[18, 31] = 1
im[40, 50] = 1

H = hab(im, -5, 5, 0.05, -100, 100, 1)

plt.figure(figsize=(10,6))
plt.imshow(H, cmap='gray', aspect='auto')
plt.title("Accumulator Space in (a, b) Domain")
plt.xlabel('Parameter b (Intercept)')
plt.ylabel('Parameter a (Slope)')
plt.show()


### Mathematical Singularities (Edge Cases)

The directional representation $y = ax + b$ is numerically unstable (singular) for layouts with vertical edges (where $x = \text{const}$).

In such cases the slope coefficient $a$ tends to infinity ($a \to \infty$), which in discrete systems would require an accumulator matrix of infinite size. This is a critical architectural limitation that production systems (e.g., OpenCV) avoid by using the trigonometric normal form (the $\rho, \theta$ space).